In [0]:
# Healthcare Gold Analytics Layer

from pyspark.sql import functions as F

catalog_schema = "workspace.healthcare"

print("Healthcare Gold Analytics Started")

In [0]:
# Read cleaned Silver tables

patients_df = spark.table(f"{catalog_schema}.silver_patients")
doctors_df = spark.table(f"{catalog_schema}.silver_doctors")
appointments_df = spark.table(f"{catalog_schema}.silver_appointments")
billing_df = spark.table(f"{catalog_schema}.silver_billing")

print("All Silver tables loaded successfully")

In [0]:
# Calculate hospital-level KPIs

patient_count = patients_df.count()
doctor_count = doctors_df.count()
appointment_count = appointments_df.count()

completed_appointments = (
    appointments_df
    .filter(F.col("Status") == "Completed")
    .count()
)

pending_appointments = (
    appointments_df
    .filter(F.col("Status") == "Pending")
    .count()
)

cancelled_appointments = (
    appointments_df
    .filter(F.col("Status") == "Cancelled")
    .count()
)

billing_summary = billing_df.agg(
    F.sum(
        F.when(F.col("PaymentStatus") == "Paid", F.col("Amount"))
         .otherwise(0)
    ).alias("TotalRevenue"),
    F.sum(
        F.when(F.col("PaymentStatus") == "Pending", F.col("Amount"))
         .otherwise(0)
    ).alias("PendingAmount")
)

gold_kpi = (
    billing_summary
    .withColumn("TotalPatients", F.lit(patient_count))
    .withColumn("TotalDoctors", F.lit(doctor_count))
    .withColumn("TotalAppointments", F.lit(appointment_count))
    .withColumn("CompletedAppointments", F.lit(completed_appointments))
    .withColumn("PendingAppointments", F.lit(pending_appointments))
    .withColumn("CancelledAppointments", F.lit(cancelled_appointments))
    .select(
        "TotalPatients",
        "TotalDoctors",
        "TotalAppointments",
        "CompletedAppointments",
        "PendingAppointments",
        "CancelledAppointments",
        "TotalRevenue",
        "PendingAmount"
    )
)

display(gold_kpi)

In [0]:
# Department-wise appointment analysis

gold_department_analytics = (
    appointments_df
    .join(doctors_df, on="DoctorID", how="inner")
    .groupBy("Department")
    .agg(
        F.countDistinct("AppointmentID").alias("TotalAppointments"),
        F.sum(
            F.when(F.col("Status") == "Completed", 1).otherwise(0)
        ).alias("CompletedAppointments"),
        F.sum(
            F.when(F.col("Status") == "Pending", 1).otherwise(0)
        ).alias("PendingAppointments"),
        F.sum(
            F.when(F.col("Status") == "Cancelled", 1).otherwise(0)
        ).alias("CancelledAppointments")
    )
    .orderBy(F.desc("TotalAppointments"))
)

display(gold_department_analytics)

In [0]:
# Doctor-wise performance analysis

gold_doctor_performance = (
    appointments_df
    .join(doctors_df, on="DoctorID", how="inner")
    .groupBy("DoctorID", "DoctorName", "Department")
    .agg(
        F.countDistinct("AppointmentID").alias("TotalAppointments"),
        F.sum(
            F.when(F.col("Status") == "Completed", 1).otherwise(0)
        ).alias("CompletedAppointments"),
        F.sum(
            F.when(F.col("Status") == "Pending", 1).otherwise(0)
        ).alias("PendingAppointments")
    )
    .orderBy(F.desc("TotalAppointments"))
)

display(gold_doctor_performance)

In [0]:
# Billing analytics

gold_billing_analytics = (
    billing_df
    .groupBy("PaymentStatus")
    .agg(
        F.count("BillingID").alias("TotalBills"),
        F.sum("Amount").alias("TotalAmount"),
        F.round(F.avg("Amount"), 2).alias("AverageBillAmount")
    )
    .orderBy("PaymentStatus")
)

display(gold_billing_analytics)

In [0]:
# Monthly appointment trends

gold_monthly_trends = (
    appointments_df
    .withColumn(
        "AppointmentMonth",
        F.date_format("AppointmentDate", "yyyy-MM")
    )
    .groupBy("AppointmentMonth")
    .agg(
        F.count("AppointmentID").alias("TotalAppointments"),
        F.sum(
            F.when(F.col("Status") == "Completed", 1).otherwise(0)
        ).alias("CompletedAppointments"),
        F.sum(
            F.when(F.col("Status") == "Pending", 1).otherwise(0)
        ).alias("PendingAppointments")
    )
    .orderBy("AppointmentMonth")
)

display(gold_monthly_trends)

In [0]:
# Patient city analysis

gold_patient_city = (
    patients_df
    .groupBy("City")
    .agg(
        F.countDistinct("PatientID").alias("TotalPatients"),
        F.round(F.avg("Age"), 2).alias("AveragePatientAge")
    )
    .orderBy(F.desc("TotalPatients"))
)

display(gold_patient_city)

In [0]:
# Save Gold analytics tables

gold_tables = {
    "gold_hospital_kpi": gold_kpi,
    "gold_department_analytics": gold_department_analytics,
    "gold_doctor_performance": gold_doctor_performance,
    "gold_billing_analytics": gold_billing_analytics,
    "gold_monthly_trends": gold_monthly_trends,
    "gold_patient_city": gold_patient_city
}

for table_name, df in gold_tables.items():

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog_schema}.{table_name}")
    )

    print(f"Successfully saved: {table_name}")

print("Gold Analytics completed successfully!")

In [0]:
# Verify all Gold tables

for table_name in gold_tables.keys():

    df = spark.table(f"{catalog_schema}.{table_name}")

    print(f"\nGold Table: {table_name}")
    print(f"Total Records: {df.count()}")

    df.show(10, truncate=False)